In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_GetDeltaWorklist
# MAGIC
# MAGIC Builds the per-table ForEach worklist from delta_sync_queue
# MAGIC for one run and one connection.

# COMMAND ----------

# MAGIC %run ../shared/_common

# COMMAND ----------

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("max_tables", "0")
dbutils.widgets.text("only_source_table_ids", "")

run_id = dbutils.widgets.get("run_id").strip() or get_run_id()
connection_id = CONNECTION_ID

try:
    max_tables = int(
        dbutils.widgets.get("max_tables").strip() or "0"
    )
except ValueError as exc:
    raise ValueError(
        "max_tables must be a non-negative integer"
    ) from exc

only_source_table_ids = {
    value.strip()
    for value in dbutils.widgets.get(
        "only_source_table_ids"
    ).split(",")
    if value.strip()
}

if not run_id:
    raise ValueError("run_id is required")

if not connection_id:
    raise ValueError("connection_id is required")

if max_tables < 0:
    raise ValueError(
        "max_tables must be greater than or equal to 0"
    )

# COMMAND ----------

def ctrl(table_name):
    return (
        f"{quote_databricks(CATALOG)}."
        f"{quote_databricks(CONTROL_SCHEMA)}."
        f"{quote_databricks(table_name)}"
    )

rows = spark.sql(
    f"""
    SELECT
        run_id,
        connection_id,
        source_table_id
    FROM {ctrl("delta_sync_queue")}
    WHERE run_id = {escape_string_literal(run_id)}
      AND connection_id =
          {escape_string_literal(connection_id)}
      AND status = 'QUEUED'
    ORDER BY source_table_id
    """
).collect()

# COMMAND ----------

worklist = []

for row in rows:
    source_table_id = row["source_table_id"]

    if (
        only_source_table_ids
        and source_table_id not in only_source_table_ids
    ):
        continue

    worklist.append(
        {
            "run_id": row["run_id"],
            "connection_id": row["connection_id"],
            "source_table_id": source_table_id,
        }
    )

if max_tables > 0:
    worklist = worklist[:max_tables]

if only_source_table_ids:
    found_ids = {
        item["source_table_id"]
        for item in worklist
    }

    missing_ids = sorted(
        only_source_table_ids - found_ids
    )

    if missing_ids:
        raise ValueError(
            "Requested source_table_id values are not QUEUED "
            "for this run and connection: "
            + ", ".join(missing_ids)
        )

# COMMAND ----------

worklist_json = json.dumps(
    worklist,
    separators=(",", ":"),
)

set_task_value(
    "worklist",
    worklist_json,
)

set_task_value(
    "worklist_count",
    len(worklist),
)

print(
    f"Delta worklist created: "
    f"run_id={run_id}, "
    f"connection_id={connection_id}, "
    f"tables={len(worklist)}"
)

dbutils.notebook.exit(
    json.dumps(
        {
            "status": "SUCCEEDED",
            "run_id": run_id,
            "connection_id": connection_id,
            "worklist_count": len(worklist),
            "worklist": worklist,
        }
    )
)